In [1]:
import os
import tarfile
import random
import re
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import _download_asset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from jiwer import wer
from tqdm.auto import tqdm
import whisper

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class Config:
    sr = 16000
    batch_size = 4
    epochs = 50
    lr = 1e-3
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tar_path = "../data/raw/ru_train_0.tar"
    tsv_path = "../data/raw/train(1).tsv"
    extract_dir = "../data/raw/ru_train_data"

    n_fft = 512
    hop_length = 256
    win_length = 512
    num_freqs = n_fft // 2 + 1
    N_neighbors = 15
    tau = 2 
    T_frames = 192

In [4]:
class FullSubNet(nn.Module):
    def __init__(self, num_freqs=257, N=15):
        super(FullSubNet, self).__init__()
        self.F = num_freqs
        self.N = N
        
        self.full_lstm = nn.LSTM(input_size=self.F, hidden_size=512, num_layers=2, batch_first=True)
        self.full_linear = nn.Linear(512, self.F)
        self.full_relu = nn.ReLU()

        self.sub_lstm = nn.LSTM(input_size=2*N + 2, hidden_size=384, num_layers=2, batch_first=True)
        self.sub_linear = nn.Linear(384, 2)

    def forward(self, x_mag):

        x_mag = x_mag.transpose(1, 2)
        B, T, F_dim = x_mag.shape

        mu_full = x_mag.mean(dim=[1, 2], keepdim=True)
        x_mag_norm = x_mag / (mu_full + 1e-8)

        full_out, _ = self.full_lstm(x_mag_norm)
        g_full = self.full_relu(self.full_linear(full_out))

        x_padded = F.pad(x_mag.transpose(1, 2), (0, 0, self.N, self.N), mode='circular')

        x_unfolded = x_padded.unfold(1, 2*self.N + 1, 1)

        g_full_expanded = g_full.transpose(1, 2).unsqueeze(-1)
        sub_input = torch.cat([x_unfolded, g_full_expanded], dim=-1)

        mu_sub = sub_input.mean(dim=2, keepdim=True)
        sub_input_norm = sub_input / (mu_sub + 1e-8)

        sub_input_reshaped = sub_input_norm.reshape(B * F_dim, T, -1)

        sub_out, _ = self.sub_lstm(sub_input_reshaped)
        cirm = self.sub_linear(sub_out)
        
        return cirm.view(B, F_dim, T, 2).transpose(1, 2)

In [5]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)
        
    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [6]:
def calculate_compressed_cirm(clean_complex, noisy_complex, K=10.0, C=0.1):

    den = noisy_complex.real**2 + noisy_complex.imag**2 + 1e-8
    mask_real = (clean_complex.real * noisy_complex.real + clean_complex.imag * noisy_complex.imag) / den
    mask_imag = (clean_complex.imag * noisy_complex.real - clean_complex.real * noisy_complex.imag) / den
    
    cirm_real_comp = K * torch.tanh((C * mask_real) / 2.0)
    cirm_imag_comp = K * torch.tanh((C * mask_imag) / 2.0)
    
    return torch.stack([cirm_real_comp, cirm_imag_comp], dim=-1).transpose(1, 2)

def apply_uncompressed_cirm(noisy_complex, cirm_compressed_pred, K=10.0, C=0.1):

    cirm_compressed_pred = torch.clamp(cirm_compressed_pred, min=-K + 1e-6, max=K - 1e-6)
    
    mask_uncompressed = (2.0 / C) * torch.atanh(cirm_compressed_pred / K)
    
    mask_real = mask_uncompressed[..., 0].transpose(1, 2)
    mask_imag = mask_uncompressed[..., 1].transpose(1, 2)
    
    noisy_real = noisy_complex.real
    noisy_imag = noisy_complex.imag
    
    est_real = noisy_real * mask_real - noisy_imag * mask_imag
    est_imag = noisy_real * mask_imag + noisy_imag * mask_real
    
    return torch.complex(est_real, est_imag)

In [7]:
if not os.path.exists(Config.extract_dir):
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar: 
        tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

import soundfile as sf
import torch

babble_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM

waveform_np, sr_b = sf.read(babble_path, always_2d=True)
BABBLE_WAVEFORM = torch.tensor(waveform_np.T, dtype=torch.float32)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM

waveform_r_np, sr_r = sf.read(rir_path, always_2d=True)
RIR_WAVEFORM = torch.tensor(waveform_r_np.T, dtype=torch.float32)

RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

In [8]:
class SpeechEnhancementDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True, max_len_sec=3.0):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        self.max_len_sec = max_len_sec
        self.files = []
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.endswith('.mp3') and f in ref_dict:
                    self.files.append(os.path.join(root, f))

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        text = self.ref_dict[filename]

        wav_np, sr = librosa.load(file_path, sr=None, mono=False)
        waveform = torch.from_numpy(wav_np)
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
            
        if sr != Config.sr:
            waveform = T.Resample(sr, Config.sr)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if self.is_train:
            max_samples = int(self.max_len_sec * Config.sr)
            if waveform.shape[-1] > max_samples:
                start = random.randint(0, waveform.shape[-1] - max_samples)
                waveform = waveform[:, start:start + max_samples]
            else:
                pad_len = max_samples - waveform.shape[-1]
                waveform = F.pad(waveform, (0, pad_len))
        
        clean = waveform
        noisy = apply_noise(clean) if self.is_train else clean

        return noisy.squeeze(0), clean.squeeze(0), text

def collate_fn(batch):
    noisy_list, clean_list, texts = zip(*batch)
    return pad_sequence(noisy_list, batch_first=True), \
           pad_sequence(clean_list, batch_first=True), list(texts)

In [9]:
class FullSubNetHybridLoss(nn.Module):
    def __init__(self, alpha=1.0, p=0.3):
        super().__init__()
        self.mse = nn.MSELoss()
        self.l1 = nn.L1Loss()
        self.alpha = alpha
        self.p = p

    def forward(self, pred_cirm, target_cirm, pred_complex_spec, target_complex_spec):
        loss_cirm = self.mse(pred_cirm, target_cirm)

        pred_mag = torch.abs(pred_complex_spec) + 1e-8
        target_mag = torch.abs(target_complex_spec) + 1e-8

        pred_mag_comp = pred_mag ** self.p
        target_mag_comp = target_mag ** self.p

        loss_mag = self.l1(pred_mag_comp, target_mag_comp)

        return self.alpha * loss_cirm + loss_mag

In [10]:
dataset = SpeechEnhancementDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size],generator = generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn)

model = FullSubNet(num_freqs=Config.num_freqs, N=Config.N_neighbors).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)
criterion = FullSubNetHybridLoss()
window = torch.hann_window(Config.n_fft).to(Config.device)

In [11]:
for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        optimizer.zero_grad()
    
        X = torch.stft(noisy, Config.n_fft, Config.hop_length, window=window, return_complex=True, center=True)
        S = torch.stft(clean, Config.n_fft, Config.hop_length, window=window, return_complex=True, center=True)
    
        pred_cirm = model(torch.abs(X)) # (B, T, F, 2)
        true_cirm = calculate_compressed_cirm(S, X)
    
        tau = Config.tau
        pred_cirm_aligned = pred_cirm[:, tau:, :, :]
        true_cirm_aligned = true_cirm[:, :-tau, :, :]
        X_aligned = X[:, :, :-tau]
        S_aligned = S[:, :, :-tau]
    
        S_hat = apply_uncompressed_cirm(X_aligned, pred_cirm_aligned)
    
        loss = criterion(pred_cirm_aligned, true_cirm_aligned, S_hat, S_aligned)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

torch.save(model.state_dict(), "../models/fullsubnet_weights.pth")

Epoch 1:   0%|          | 1/5954 [00:11<18:31:29, 11.20s/it, Loss=0.3639]


KeyboardInterrupt: 

In [13]:
model = FullSubNet(num_freqs=Config.num_freqs, N=Config.N_neighbors).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)
criterion = FullSubNetHybridLoss()

model.load_state_dict(torch.load('../models/fullsubnet_weights.pth', map_location='cpu'))

<All keys matched successfully>

In [14]:
seed_everything(42)

In [ ]:
def evaluate_fullsubnet(model, device, val_dataset, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
    
    window = torch.hann_window(Config.n_fft).to(device)

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Eval"):
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)
            
            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav.unsqueeze(0), force_type=n_type, file_seed=idx).to(device)
                
                X = torch.stft(noisy_wav.squeeze(1), Config.n_fft, Config.hop_length, window=window, return_complex=True)
                
                pred_cirm = model(torch.abs(X))
                
                tau = Config.tau
                pred_aligned = pred_cirm[:, tau:, :, :]
                X_aligned = X[:, :, :-tau]
                S_hat = apply_uncompressed_cirm(X_aligned, pred_aligned)
                
                denoised_wav = torch.istft(S_hat, Config.n_fft, Config.hop_length, window=window)
                
                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate_fullsubnet(model, Config.device, val_ds, limit = 20)

WER Eval:   0%|          | 0/20 [00:00<?, ?it/s]


Noise Type | WER Noisy  | WER Denoised
babble     | 0.5376     | 0.4233    
rir        | 1.0000     | 1.0214    
white      | 0.5670     | 0.6735    


In [ ]:
from jiwer import process_words
import numpy as np
import torch
import whisper
from tqdm.auto import tqdm

def evaluate_fullsubnet_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
        
    window = torch.hann_window(Config.n_fft).to(device)
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_tensor_cpu = clean_wav.unsqueeze(0)
            
            for n_type in noise_types:
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_wav = noisy_tensor_cpu.to(device)
                
                X = torch.stft(noisy_wav.squeeze(1), Config.n_fft, Config.hop_length, window=window, return_complex=True)
                
                pred_cirm = model(torch.abs(X))
                
                tau = Config.tau
                pred_aligned = pred_cirm[:, tau:, :, :]
                X_aligned = X[:, :, :-tau]
                S_hat = apply_uncompressed_cirm(X_aligned, pred_aligned)
                
                denoised_wav = torch.istft(S_hat, Config.n_fft, Config.hop_length, window=window)
                
                noisy_np = noisy_wav.squeeze().cpu().numpy()
                denoised_np = denoised_wav.squeeze().cpu().numpy()
                
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
                
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
                
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)
                    
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)
                    
    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 65)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
        
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
        
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
        
        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")

seed_everything(42)
evaluate_fullsubnet_components(model, Config.device, val_ds, limit = 20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
-----------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.4233 (0.0586/0.3601/0.0045) | 0.1143  
rir      | 1.0000 (0.3116/0.6884/0.0000) | 1.0214 (0.3414/0.6586/0.0214) | -0.0214 
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.6735 (0.2509/0.3821/0.0406) | -0.1065 
